# Libraries management

In [ ]:
# # import cx_Oracle
# import oracledb
# import sys

# # 1. Enable 'Thin' mode (no Oracle Client needed)
# # 2. Alias oracledb to cx_Oracle so your existing code works
# oracledb.version = "8.3.0" # Fakes the version cx_Oracle expects
# sys.modules["cx_Oracle"] = oracledb

# # Now your existing line will work without errors:
# import cx_Oracle

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import win32com.client as win32
# import gc as gc
# from time import sleep
# # import sys as sys
# import os as os
# from datetime import datetime as datetime
# import pythoncom as pythoncom
# # from minio import Minio
# # import smtplib
# ## Check python version
# import sys
# import platform

# from dotenv import load_dotenv
# load_dotenv()

In [ ]:
# convert.py — turns every monthly .xlsx into a UTF-8 CSV + a BigQuery schema.json
import re, json, unicodedata
from pathlib import Path
import pandas as pd

# ── EDIT THIS to the folder holding your .xlsx files ──────────────────────
SRC = Path(r"D:\OneDrive - Central Group\Stella's files - 1. HAND OVER\1. CAMPAIGN MANAGEMENT\HYPER GO! & Big C\6. VOUCHER\Voucher_ops_raw")
# ──────────────────────────────────────────────────────────────────────────
OUT = SRC / "csv"
OUT.mkdir(exist_ok=True)

def bq_name(col):                      # make a clean, SQL-friendly column name
    s = unicodedata.normalize("NFKD", str(col).strip())
    s = s.encode("ascii", "ignore").decode("ascii")   # drop VN accents
    s = re.sub(r"[^0-9A-Za-z_]+", "_", s).strip("_").lower()
    return s if s and not s[0].isdigit() else "col_" + s

template = None
for xlsx in sorted(SRC.glob("*.xlsx")):
    if xlsx.stem.startswith("~$"):     # skip Excel lock files
        continue
    month = xlsx.stem.split("_")[0]                    # "2024-12" from the filename
    df = pd.read_excel(xlsx, sheet_name=0, dtype=str).fillna("")   # read ALL as text
    df["source_month"] = month                         # so every row knows its file
    if template is None:
        template = list(df.columns)
    elif list(df.columns) != template:
        print(f"  !! {xlsx.name}: columns differ — check this file")
        df = df.reindex(columns=template, fill_value="")
    df.to_csv(OUT / f"{xlsx.stem}.csv", index=False, encoding="utf-8")
    print(f"  {xlsx.name:28} {len(df):>8,} rows")

# auto-build a BigQuery schema (everything STRING = safe landing zone), with de-dup
seen, names = {}, []
for c in template:
    n = bq_name(c)
    if n in seen:
        seen[n] += 1; n = f"{n}_{seen[n]}"
    else:
        seen[n] = 0
    names.append(n)
(OUT / "schema.json").write_text(
    json.dumps([{"name": n, "type": "STRING"} for n in names], indent=2), encoding="utf-8")
print(f"\nDone → {OUT}\nColumns: {', '.join(names)}")